# AlphaTrain Path B: V12 policy + oracle correction auxiliary

Continuing past pillar2z (mean 6,952 policy-only). Strategy: re-train fresh from the same pillar2y2 warm-start, with color augmentation and a small **conditional-KL oracle correction loss** mined from K=32 rollouts on 16,897 crisis anchors.

## Ablation matrix (runs in this notebook)

| run | --oracle-lambda | --color-augment | purpose |
|---|---|---|---|
| **B** | 0.0  | yes | Isolate gain from color symmetry alone (sanity check vs pillar2z) |
| **C** | 0.05 | yes | Conservative oracle pressure |
| **D** | 0.10 | yes | Aggressive oracle pressure |

Run **B first**. If B underperforms pillar2z, debug the new pipeline before queueing C/D — the oracle additions cannot be the cause when λ=0.

## Loss

    loss = L_v12_policy_CE + λ · weighted_mean( KL( softmax(β · cap_rates_top6) ‖ softmax(gather(logits, actions_top6)) ) )
    w(Δcap) = clip( (Δcap − 0.05) / 0.20, 0, 1 ) ** 2
    β = 10

Reliability weight maps onto our histogram cleanly: 33% noise-floor anchors get w≈0, 11% high-margin (Δcap ≥ 0.15) get w∈[0.25, 1].

## Upload to Drive (`MyDrive/alphatrain/`)

1. `colorlines_path_b.tar.gz` — code + oracle tensor (~1.1 MB)
2. `v12_pillar2z.pt.gz` — V12 tensor, already there from pillar2z run (~2-3 GB)
3. `pillar2y2_epoch_40.pt` — warm-start, already there (~143 MB)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_path_b.tar.gz /content/
!cd /content && tar xzf colorlines_path_b.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar2y2_epoch_40.pt',
            '/content/alphatrain/data/model.pt')
print(f'Warm-start: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

t0 = time.time()
!gzip -dc {DRIVE}/v12_pillar2z.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'V12 tensor: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

print(f'Oracle tensor: {os.path.getsize("/content/alphatrain/data/phase1_oracle_path_b.pt")/1e6:.1f} MB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

%cd /content
!python -m pytest alphatrain/tests/test_train_path_b.py -v --tb=short 2>&1 | tail -20

## Run B — V12 + color aug, no oracle (sanity)

λ=0 → oracle tensor not loaded. Compare best-epoch val_loss + 100-seed gameplay eval vs pillar2z. If B does not match pillar2z (within fp16 noise on val_loss), pipeline has a bug — stop before C/D.

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --color-augment \
    --epochs 40 --batch-size 32768 --lr 3e-4 --warmup-epochs 2 \
    --oracle-lambda 0.0 \
    --copy-to /content/drive/MyDrive/alphatrain/path_b_B_best.pt \
    --save-dir /content/checkpoints/path_b_B

In [ ]:
# Copy every Run B epoch checkpoint to Drive (for retrospective eval)
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'path_b_B'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')

## Run C — λ=0.05 (conservative oracle)

**Only run if B passes.** Reuses the same V12 corpus and warm-start; only difference vs B is oracle aux loss enabled. Tracks oracle val metrics (KL all/weighted, top1 by margin bucket) each epoch.

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --oracle-tensor alphatrain/data/phase1_oracle_path_b.pt \
    --oracle-lambda 0.05 --oracle-beta 10.0 \
    --oracle-noise-floor 0.05 --oracle-scale 0.20 \
    --oracle-batch-size 4096 \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --color-augment \
    --epochs 40 --batch-size 32768 --lr 3e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/path_b_C_best.pt \
    --save-dir /content/checkpoints/path_b_C

In [ ]:
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'path_b_C'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')

## Run D — λ=0.10 (aggressive oracle)

**Only run if B passes.** Same as C but 2× oracle pressure.

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --oracle-tensor alphatrain/data/phase1_oracle_path_b.pt \
    --oracle-lambda 0.10 --oracle-beta 10.0 \
    --oracle-noise-floor 0.05 --oracle-scale 0.20 \
    --oracle-batch-size 4096 \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --color-augment \
    --epochs 40 --batch-size 32768 --lr 3e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/path_b_D_best.pt \
    --save-dir /content/checkpoints/path_b_D

In [ ]:
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'path_b_D'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')